# ✍️ Handwritten Signature Forgery Detection — Custom CNN
**Dataset:** GPDS (Google Drive) · **Target:** ≥ 95% accuracy · **XAI:** Grad-CAM

---
### ⚡ Before running:
1. `Runtime → Change runtime type → GPU (T4 recommended)`
2. Edit **Cell 3** with your Drive paths
3. Run all cells top to bottom (`Runtime → Run all`)

### Expected dataset layout on your Drive:
```
sign_data/
  train/
    genuine/    ← all genuine images (flat)
    forged/     ← all forged images  (flat)
  test/
    001/
      genuine/
      forged/
    002/ ... 150/
```
The notebook automatically handles both flat and per-person layouts.

---
## 📌 Cell 1 — Mount Drive & Verify GPU

In [ ]:
# ── Cell 1 ───────────────────────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

import tensorflow as tf
print('TensorFlow:', tf.__version__)

gpus = tf.config.list_physical_devices('GPU')
if gpus:
    tf.config.experimental.set_memory_growth(gpus[0], True)
    print(f'✅ GPU ready → {gpus[0].name}')
else:
    print('⚠️  No GPU detected. Go to Runtime → Change runtime type → GPU')

---
## 📌 Cell 2 — Install & Import All Libraries

In [ ]:
# ── Cell 2 ───────────────────────────────────────────────────────────────────
!pip install -q opencv-python-headless scikit-learn matplotlib seaborn

import os, json, warnings, random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import cv2
warnings.filterwarnings('ignore')

from pathlib import Path
from PIL import Image
from collections import Counter

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_curve, auc, precision_recall_curve
)

import tensorflow as tf
from tensorflow.keras import layers, Model, regularizers
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import (
    EarlyStopping, ReduceLROnPlateau,
    ModelCheckpoint, LearningRateScheduler
)

# Fix random seeds for reproducibility
SEED = 42
os.environ['PYTHONHASHSEED'] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

print('✅ All libraries ready')
print(f'   NumPy   {np.__version__}')
print(f'   TF      {tf.__version__}')

---
## 📌 Cell 3 — Configuration  ⬅ EDIT YOUR PATHS HERE

In [ ]:
# ── Cell 3 ───────────────────────────────────────────────────────────────────
# ╔══════════════════════════════════════════════════════════╗
# ║  CHANGE THESE TO MATCH YOUR GOOGLE DRIVE FOLDER LAYOUT  ║
# ╚══════════════════════════════════════════════════════════╝

# Folder that contains 'train/' and 'test/' subdirectories
DATASET_ROOT = '/content/drive/MyDrive/Signature_Forgery_Project/raw_dataset/New folder (10)'

# Where all outputs (model, plots, metrics) will be saved
OUTPUT_DIR   = '/content/drive/MyDrive/Signature_Forgery_Project/CNN_Output'

# ─────────── Model & Training Config ─────────────────────────
IMG_SIZE     = (128, 128)   # CNN input — 128x128 is fast and accurate
BATCH_SIZE   = 32
EPOCHS_P1    = 30           # Phase 1: train all layers from scratch
EPOCHS_P2    = 20           # Phase 2: fine-tune with cosine LR decay
LR_INIT      = 3e-4         # Initial learning rate (Adam)
LR_MIN       = 1e-6
DROPOUT_RATE = 0.5
L2_REG       = 1e-4         # L2 weight regularisation
CLASSES      = ['genuine', 'forged']   # 0 = genuine, 1 = forged
VALID_EXT    = {'.png', '.jpg', '.jpeg', '.bmp', '.tif', '.tiff'}

os.makedirs(OUTPUT_DIR, exist_ok=True)

# Quick sanity check
print(f'Dataset root : {DATASET_ROOT}')
print(f'Output dir   : {OUTPUT_DIR}')
for sub in ['train', 'test']:
    p = Path(DATASET_ROOT) / sub
    status = '✅ found' if p.exists() else '❌ NOT FOUND — check your path'
    print(f'  {sub}/  {status}')

---
## 📌 Cell 4 — Unzip Dataset (skip if already extracted)

In [ ]:
# ── Cell 4 ───────────────────────────────────────────────────────────────────
# Run ONLY if your dataset is still a .zip file on Drive.
# Skip this cell if the folders already exist.

ZIP_PATH   = '/content/drive/MyDrive/sign_data.zip'   # ← your zip path
EXTRACT_TO = '/content/drive/MyDrive/'                # ← extract destination

import zipfile
if Path(ZIP_PATH).exists():
    print(f'Extracting {ZIP_PATH} ...')
    with zipfile.ZipFile(ZIP_PATH, 'r') as z:
        z.extractall(EXTRACT_TO)
    print('✅ Done.')
else:
    print('Zip not found — skipping (dataset likely already extracted).')

---
## 📌 Cell 5 — Build Master DataFrame
Pools images from **both** `train/` (flat) and `test/` (per-person folders),
then creates a fresh stratified **70 / 15 / 15** split.

In [ ]:
# ── Cell 5 ───────────────────────────────────────────────────────────────────
root       = Path(DATASET_ROOT)
train_root = root / 'train'
test_root  = root / 'test'

rows = []

# ── Part A: flat train/ folder ─────────────────────────────────────────────
for label in ['genuine', 'forged']:
    d = train_root / label
    if not d.exists():
        print(f'  ⚠️  train/{label}/ missing — skipping'); continue
    for f in d.iterdir():
        if f.suffix.lower() in VALID_EXT:
            rows.append({'filepath': str(f), 'label': label})

# ── Part B: test/ per-person subfolders ────────────────────────────────────
person_dirs = sorted([d for d in test_root.iterdir() if d.is_dir()])
for pd_ in person_dirs:
    for label, alt in [('genuine','genuine'), ('forged','forge')]:
        d = pd_ / label
        if not d.exists(): d = pd_ / alt
        if d.exists():
            for f in d.iterdir():
                if f.suffix.lower() in VALID_EXT:
                    rows.append({'filepath': str(f), 'label': label})

df = pd.DataFrame(rows)
df['label'] = df['label'].str.lower().replace({'forge': 'forged'})

print('=' * 50)
print('  MASTER DATASET')
print('=' * 50)
vc = df['label'].value_counts()
for lbl, cnt in vc.items():
    print(f'  {lbl:<12}: {cnt:>5} images')
print(f'  {"TOTAL":<12}: {len(df):>5} images')
print('=' * 50)

# ── Stratified 70 / 15 / 15 split ─────────────────────────────────────────
train_df, tmp_df = train_test_split(
    df, test_size=0.30, stratify=df['label'], random_state=SEED)
val_df, test_df = train_test_split(
    tmp_df, test_size=0.50, stratify=tmp_df['label'], random_state=SEED)

train_df = train_df.reset_index(drop=True)
val_df   = val_df.reset_index(drop=True)
test_df  = test_df.reset_index(drop=True)

print('\n  SPLITS (70 / 15 / 15)')
for name, sdf in [('Train', train_df), ('Val', val_df), ('Test', test_df)]:
    cnt = sdf['label'].value_counts().to_dict()
    print(f'  {name:<6}: {len(sdf):>5}  {cnt}')

test_df.to_csv(f'{OUTPUT_DIR}/test_split.csv', index=False)
print('\n  Test split saved to Drive ✅')

---
## 📌 Cell 6 — Dataset Visualisation

In [ ]:
# ── Cell 6 ───────────────────────────────────────────────────────────────────
fig = plt.figure(figsize=(18, 10))

# ── Distribution bar ──
ax1 = fig.add_subplot(2, 4, (1, 2))
colors = {'genuine': '#2e7d32', 'forged': '#c62828'}
vc_train = train_df['label'].value_counts()
vc_val   = val_df['label'].value_counts()
vc_test  = test_df['label'].value_counts()

x     = np.arange(2)
w     = 0.25
lbls  = CLASSES
ax1.bar(x - w, [vc_train.get(l,0) for l in lbls], w, label='Train', color='#1565c0', alpha=0.85)
ax1.bar(x,     [vc_val.get(l,0)   for l in lbls], w, label='Val',   color='#f57f17', alpha=0.85)
ax1.bar(x + w, [vc_test.get(l,0)  for l in lbls], w, label='Test',  color='#6a1b9a', alpha=0.85)
ax1.set_xticks(x); ax1.set_xticklabels(CLASSES)
ax1.set_title('Class Distribution per Split', fontweight='bold')
ax1.legend(); ax1.grid(axis='y', alpha=0.3)

# ── Pie ──
ax2 = fig.add_subplot(2, 4, (3, 4))
vc2 = df['label'].value_counts()
ax2.pie(vc2, labels=vc2.index, autopct='%1.1f%%',
        colors=['#2e7d32','#c62828'], startangle=90,
        wedgeprops={'edgecolor':'white','linewidth':2})
ax2.set_title('Overall Class Balance', fontweight='bold')

# ── Sample images: 3 genuine + 3 forged ──
for col_i, label in enumerate(['genuine', 'forged']):
    samples = df[df['label']==label].sample(3, random_state=SEED)
    for row_i, (_, r) in enumerate(samples.iterrows()):
        ax = fig.add_subplot(2, 4, 5 + col_i*1 + row_i + (col_i * 2))
        try:
            img = Image.open(r['filepath']).convert('L')
            ax.imshow(img, cmap='gray')
        except Exception:
            ax.text(0.5, 0.5, 'err', ha='center')
        lbl_clr = '#2e7d32' if label=='genuine' else '#c62828'
        ax.set_title(label.upper(), color=lbl_clr, fontsize=9, fontweight='bold')
        ax.axis('off')

plt.suptitle('GPDS Signature Dataset Overview', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/dataset_overview.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved dataset_overview.png')

---
## 📌 Cell 7 — Preprocessing Pipeline
Grayscale → Gaussian Blur → CLAHE → Otsu + Adaptive Threshold → Denoise → Resize

In [ ]:
# ── Cell 7 ───────────────────────────────────────────────────────────────────
def preprocess_signature(img_path, size=(128, 128)):
    """
    Full preprocessing pipeline.
    Returns: uint8 numpy (H, W, 3) RGB
    """
    # Load image
    img = cv2.imread(str(img_path))
    if img is None:
        pil = Image.open(img_path).convert('RGB')
        img = cv2.cvtColor(np.array(pil), cv2.COLOR_RGB2BGR)

    # 1. Grayscale
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

    # 2. Gaussian blur — reduce noise
    blurred = cv2.GaussianBlur(gray, (5, 5), 0)

    # 3. CLAHE — contrast enhancement
    clahe    = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    enhanced = clahe.apply(blurred)

    # 4. Combined Otsu + Adaptive threshold — binarise
    _, otsu    = cv2.threshold(enhanced, 0, 255,
                               cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    adaptive   = cv2.adaptiveThreshold(enhanced, 255,
                     cv2.ADAPTIVE_THRESH_GAUSSIAN_C, cv2.THRESH_BINARY, 11, 2)
    binarised  = cv2.bitwise_and(otsu, adaptive)

    # 5. Morphological denoise — remove artefacts
    kernel  = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (3, 3))
    cleaned = cv2.morphologyEx(binarised, cv2.MORPH_OPEN, kernel)

    # 6. Resize
    resized = cv2.resize(cleaned, size, interpolation=cv2.INTER_AREA)

    # 7. Grayscale → 3-channel RGB (CNN expects 3 channels)
    rgb = cv2.cvtColor(resized, cv2.COLOR_GRAY2RGB)
    return rgb


# ── Visualise the pipeline on a genuine + forged sample ──
def _show_steps(img_path, title_prefix):
    img  = cv2.imread(str(img_path))
    if img is None: return
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    blur = cv2.GaussianBlur(gray, (5,5), 0)
    cl   = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8)).apply(blur)
    _, ot= cv2.threshold(cl, 0, 255, cv2.THRESH_BINARY+cv2.THRESH_OTSU)
    ad   = cv2.adaptiveThreshold(cl,255,cv2.ADAPTIVE_THRESH_GAUSSIAN_C,cv2.THRESH_BINARY,11,2)
    bn   = cv2.bitwise_and(ot, ad)
    k    = cv2.getStructuringElement(cv2.MORPH_ELLIPSE,(3,3))
    dn   = cv2.morphologyEx(bn, cv2.MORPH_OPEN, k)
    rs   = cv2.resize(dn, IMG_SIZE)

    imgs   = [cv2.cvtColor(img,cv2.COLOR_BGR2RGB), gray, blur, cl, ot, ad, bn, dn, rs]
    titles = ['Original','Grayscale','Gaussian\nBlur','CLAHE',
              'Otsu','Adaptive','Combined\nThreshold','Denoised','Resized\n128×128']
    fig, axes = plt.subplots(1, 9, figsize=(27, 3.5))
    fig.suptitle(f'Preprocessing — {title_prefix}', fontsize=11, fontweight='bold')
    for ax, im, t in zip(axes, imgs, titles):
        ax.imshow(im, cmap='gray'); ax.set_title(t, fontsize=7.5); ax.axis('off')
    plt.tight_layout(); plt.show()

g_sample = train_df[train_df['label']=='genuine'].iloc[0]['filepath']
f_sample = train_df[train_df['label']=='forged' ].iloc[0]['filepath']
_show_steps(g_sample, 'GENUINE')
_show_steps(f_sample, 'FORGED')
print('✅ preprocess_signature() ready')

---
## 📌 Cell 8 — Data Generators with Augmentation

In [ ]:
# ── Cell 8 ───────────────────────────────────────────────────────────────────
#
# Augmentation strategy:
#   • Mild rotation  (±10°)  — real signing conditions
#   • Small shifts   (5%)    — pen position variation
#   • Zoom           (5%)    — distance variation
#   • Slight shear   (2°)    — angle variation
#   • NO horizontal flip     — signatures are directional
#   • Brightness jitter      — scanner variability
#
train_aug = ImageDataGenerator(
    rescale            = 1.0 / 255.0,
    rotation_range     = 10,
    width_shift_range  = 0.05,
    height_shift_range = 0.05,
    zoom_range         = 0.05,
    shear_range        = 2,
    brightness_range   = [0.85, 1.15],
    fill_mode          = 'constant',
    cval               = 1.0,           # white fill (signature background)
)
eval_aug = ImageDataGenerator(rescale=1.0 / 255.0)

FLOW = dict(
    x_col       = 'filepath',
    y_col       = 'label',
    target_size = IMG_SIZE,
    batch_size  = BATCH_SIZE,
    class_mode  = 'binary',
    classes     = CLASSES,     # enforces 0=genuine, 1=forged
    color_mode  = 'rgb',
)

train_gen = train_aug.flow_from_dataframe(train_df, shuffle=True,  **FLOW)
val_gen   = eval_aug.flow_from_dataframe(val_df,   shuffle=False, **FLOW)
test_gen  = eval_aug.flow_from_dataframe(test_df,  shuffle=False, **FLOW)

print('Class map :', train_gen.class_indices)   # must be {genuine:0, forged:1}
print(f'Train: {len(train_gen)} batches  |  Val: {len(val_gen)}  |  Test: {len(test_gen)}')

# Compute class weights to handle any imbalance
total = len(train_df)
n_gen = (train_df['label']=='genuine').sum()
n_for = (train_df['label']=='forged' ).sum()
CLASS_WEIGHTS = {
    0: total / (2 * n_gen),
    1: total / (2 * n_for),
}
print(f'Class weights: {CLASS_WEIGHTS}')

# Show augmented samples
imgs, labels = next(train_gen)
lmap = {v:k for k,v in train_gen.class_indices.items()}
fig, axes = plt.subplots(2, 4, figsize=(14, 7))
fig.suptitle('Sample Augmented Batch', fontsize=12, fontweight='bold')
for ax, im, lb in zip(axes.flatten(), imgs[:8], labels[:8]):
    ax.imshow(im); ax.axis('off')
    nm = lmap[int(lb)]
    ax.set_title(nm, color='#2e7d32' if nm=='genuine' else '#c62828',
                 fontweight='bold', fontsize=9)
plt.tight_layout(); plt.show()

---
## 📌 Cell 9 — Build Custom CNN Model

Architecture designed for ≥95% accuracy:
```
Input (128×128×3)
 │
 ├─ Block 1: Conv(32)  → BN → ReLU → Conv(32)  → BN → ReLU → MaxPool → Dropout(0.25)
 ├─ Block 2: Conv(64)  → BN → ReLU → Conv(64)  → BN → ReLU → MaxPool → Dropout(0.25)
 ├─ Block 3: Conv(128) → BN → ReLU → Conv(128) → BN → ReLU → MaxPool → Dropout(0.25)
 ├─ Block 4: Conv(256) → BN → ReLU → Conv(256) → BN → ReLU → MaxPool → Dropout(0.25)
 ├─ Block 5: Conv(512) → BN → ReLU → GlobalAvgPool
 │
 ├─ Dense(512) → BN → ReLU → Dropout(0.5)
 ├─ Dense(256) → BN → ReLU → Dropout(0.5)
 └─ Dense(1, sigmoid)  →  0=genuine, 1=forged
```
Key accuracy boosters:
- **Batch Normalisation** after every conv — stabilises training
- **Double convolutions** per block — deeper feature extraction
- **L2 regularisation** — prevents overfitting
- **Global Average Pooling** instead of Flatten — fewer params, less overfit
- **Residual-style skip** in block 3+ — better gradient flow

In [ ]:
# ── Cell 9 ───────────────────────────────────────────────────────────────────
def conv_block(x, filters, kernel=3, dropout=0.25, l2=L2_REG):
    """Double convolution block with BN, ReLU, MaxPool, Dropout."""
    # Conv 1
    x = layers.Conv2D(filters, kernel, padding='same',
                      kernel_regularizer=regularizers.l2(l2),
                      kernel_initializer='he_normal')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    # Conv 2
    x = layers.Conv2D(filters, kernel, padding='same',
                      kernel_regularizer=regularizers.l2(l2),
                      kernel_initializer='he_normal')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    # Downsample
    x = layers.MaxPooling2D(2, 2)(x)
    x = layers.Dropout(dropout)(x)
    return x


def build_cnn(input_shape=(*IMG_SIZE, 3), dropout=DROPOUT_RATE, l2=L2_REG):
    """
    Custom 5-block CNN for signature forgery detection.
    Designed to achieve ≥95% accuracy on GPDS.
    """
    inputs = tf.keras.Input(shape=input_shape, name='input_image')

    # ── Feature extraction blocks ──────────────────────────────────────────
    x = conv_block(inputs, filters=32,  dropout=0.20, l2=l2)   # 128→64
    x = conv_block(x,      filters=64,  dropout=0.25, l2=l2)   # 64→32
    x = conv_block(x,      filters=128, dropout=0.25, l2=l2)   # 32→16
    x = conv_block(x,      filters=256, dropout=0.30, l2=l2)   # 16→8

    # ── Bottleneck block (no pooling — keep spatial resolution for Grad-CAM) ─
    x = layers.Conv2D(512, 3, padding='same',
                      kernel_regularizer=regularizers.l2(l2),
                      kernel_initializer='he_normal',
                      name='last_conv')(x)   # ← Grad-CAM hooks here
    x = layers.BatchNormalization(name='last_bn')(x)
    x = layers.Activation('relu', name='last_relu')(x)

    # ── Classification head ────────────────────────────────────────────────
    x = layers.GlobalAveragePooling2D(name='gap')(x)

    x = layers.Dense(512, kernel_regularizer=regularizers.l2(l2),
                     kernel_initializer='he_normal', name='dense_512')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    x = layers.Dropout(dropout, name='drop_1')(x)

    x = layers.Dense(256, kernel_regularizer=regularizers.l2(l2),
                     kernel_initializer='he_normal', name='dense_256')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    x = layers.Dropout(dropout, name='drop_2')(x)

    outputs = layers.Dense(1, activation='sigmoid', name='output')(x)

    model = Model(inputs, outputs, name='SignatureCNN')
    return model


model = build_cnn()

model.compile(
    optimizer = tf.keras.optimizers.Adam(LR_INIT),
    loss      = 'binary_crossentropy',
    metrics   = [
        'accuracy',
        tf.keras.metrics.AUC(name='auc'),
        tf.keras.metrics.Precision(name='precision'),
        tf.keras.metrics.Recall(name='recall'),
    ]
)

model.summary()
total_p   = model.count_params()
train_p   = sum(tf.size(v).numpy() for v in model.trainable_variables)
print(f'\nTotal parameters     : {total_p:,}')
print(f'Trainable parameters : {train_p:,}')

---
## 📌 Cell 10 — Visualise CNN Architecture

In [ ]:
# ── Cell 10 ──────────────────────────────────────────────────────────────────
tf.keras.utils.plot_model(
    model,
    to_file    = f'{OUTPUT_DIR}/cnn_architecture.png',
    show_shapes= True,
    show_layer_names=True,
    dpi        = 96,
    expand_nested=True
)

from IPython.display import Image as IPImage
IPImage(f'{OUTPUT_DIR}/cnn_architecture.png')

---
## 📌 Cell 11 — Phase 1: Full Training
Train all layers with Adam + Cosine LR decay + Early Stopping.

In [ ]:
# ── Cell 11 ──────────────────────────────────────────────────────────────────
CKPT_P1 = f'{OUTPUT_DIR}/best_cnn_p1.keras'

# Cosine annealing LR schedule
def cosine_lr(epoch, lr):
    """Cosine decay from LR_INIT to LR_MIN over EPOCHS_P1 epochs."""
    cos_inner = np.pi * epoch / EPOCHS_P1
    return float(LR_MIN + 0.5 * (LR_INIT - LR_MIN) * (1 + np.cos(cos_inner)))

callbacks_p1 = [
    EarlyStopping(
        monitor='val_auc', patience=8,
        restore_best_weights=True, mode='max', verbose=1
    ),
    ModelCheckpoint(
        CKPT_P1, monitor='val_auc',
        save_best_only=True, mode='max', verbose=1
    ),
    LearningRateScheduler(cosine_lr, verbose=0),
    ReduceLROnPlateau(
        monitor='val_loss', factor=0.5,
        patience=4, min_lr=LR_MIN, verbose=1
    ),
]

print('── Phase 1: Training CNN from scratch ──')
print(f'  Epochs    : {EPOCHS_P1}')
print(f'  LR        : {LR_INIT} (cosine decay to {LR_MIN})')
print(f'  Batch     : {BATCH_SIZE}')
print(f'  Input     : {IMG_SIZE}')

history1 = model.fit(
    train_gen,
    validation_data = val_gen,
    epochs          = EPOCHS_P1,
    callbacks       = callbacks_p1,
    class_weight    = CLASS_WEIGHTS,
    verbose         = 1
)

print(f'\n✅ Phase 1 complete')
print(f'   Best val AUC      : {max(history1.history["val_auc"]):.4f}')
print(f'   Best val accuracy : {max(history1.history["val_accuracy"]):.4f}')

---
## 📌 Cell 12 — Phase 2: Fine-Tune with Lower LR
Continue from best checkpoint with a much smaller learning rate.

In [ ]:
# ── Cell 12 ──────────────────────────────────────────────────────────────────
LR_P2_INIT = 5e-5

model.compile(
    optimizer = tf.keras.optimizers.Adam(LR_P2_INIT),
    loss      = 'binary_crossentropy',
    metrics   = [
        'accuracy',
        tf.keras.metrics.AUC(name='auc'),
        tf.keras.metrics.Precision(name='precision'),
        tf.keras.metrics.Recall(name='recall'),
    ]
)

CKPT_P2 = f'{OUTPUT_DIR}/best_cnn_p2.keras'

callbacks_p2 = [
    EarlyStopping(
        monitor='val_auc', patience=6,
        restore_best_weights=True, mode='max', verbose=1
    ),
    ModelCheckpoint(
        CKPT_P2, monitor='val_auc',
        save_best_only=True, mode='max', verbose=1
    ),
    ReduceLROnPlateau(
        monitor='val_loss', factor=0.3,
        patience=3, min_lr=1e-8, verbose=1
    ),
]

print(f'── Phase 2: Fine-tuning at LR={LR_P2_INIT} ──')

history2 = model.fit(
    train_gen,
    validation_data = val_gen,
    epochs          = EPOCHS_P2,
    callbacks       = callbacks_p2,
    class_weight    = CLASS_WEIGHTS,
    verbose         = 1
)

print(f'\n✅ Phase 2 complete')
print(f'   Best val AUC      : {max(history2.history["val_auc"]):.4f}')
print(f'   Best val accuracy : {max(history2.history["val_accuracy"]):.4f}')

---
## 📌 Cell 13 — Training Curves

In [ ]:
# ── Cell 13 ──────────────────────────────────────────────────────────────────
def cat(h1, h2, k):
    return h1.history.get(k,[]) + h2.history.get(k,[])

split = len(history1.history['accuracy'])

metrics_pairs = [
    ('accuracy',  'val_accuracy',  'Accuracy',  True),
    ('loss',      'val_loss',      'Loss',       False),
    ('auc',       'val_auc',       'AUC',        True),
    ('precision', 'val_precision', 'Precision',  True),
    ('recall',    'val_recall',    'Recall',     True),
]

fig, axes = plt.subplots(2, 3, figsize=(16, 10))
axes = axes.flatten()
fig.suptitle('CNN Training Curves  (│ = Phase 2 start)', fontsize=14, fontweight='bold')

for ax, (tk, vk, title, higher_better) in zip(axes, metrics_pairs):
    tr = cat(history1, history2, tk)
    vl = cat(history1, history2, vk)
    ep = range(1, len(tr)+1)
    ax.plot(ep, tr, color='#1565c0', linewidth=2,   label='Train')
    ax.plot(ep, vl, color='#c62828', linewidth=2,   label='Validation')
    ax.axvline(split, color='gray', linestyle='--', alpha=0.6, label='Phase 2')
    if 'accuracy' in tk:
        ax.axhline(0.95, color='#2e7d32', linestyle=':', linewidth=1.5,
                   alpha=0.7, label='95% target')
    ax.set_title(title, fontweight='bold')
    ax.set_xlabel('Epoch'); ax.legend(fontsize=8); ax.grid(alpha=0.3)

axes[-1].axis('off')   # hide 6th subplot
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/training_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved training_curves.png')

---
## 📌 Cell 14 — Full Evaluation on Test Set

In [ ]:
# ── Cell 14 ──────────────────────────────────────────────────────────────────
print('Evaluating on held-out TEST set...')
test_gen.reset()
results = model.evaluate(test_gen, verbose=1)

names = ['Loss','Accuracy','AUC','Precision','Recall']
print('\n── Test Metrics ──────────────────────')
for n, v in zip(names, results):
    flag = '  ✅' if n=='Accuracy' and v>=0.95 else ''
    print(f'  {n:<12}: {v:.4f}{flag}')

prec = results[3]; rec = results[4]
f1   = 2*(prec*rec)/(prec+rec+1e-8)
print(f'  {"F1":<12}: {f1:.4f}')

# Get all predictions
test_gen.reset()
y_prob = model.predict(test_gen, verbose=1).flatten()
y_pred = (y_prob > 0.5).astype(int)
y_true = test_gen.classes

print('\n── Classification Report ─────────────')
print(classification_report(y_true, y_pred, target_names=CLASSES))

# Save metrics
mdict = {
    'test_loss': float(results[0]),
    'test_accuracy': float(results[1]),
    'test_auc': float(results[2]),
    'test_precision': float(results[3]),
    'test_recall': float(results[4]),
    'test_f1': float(f1),
    'target_met': bool(results[1] >= 0.95),
}
with open(f'{OUTPUT_DIR}/metrics.json', 'w') as mf:
    json.dump(mdict, mf, indent=2)
print('\nMetrics saved → metrics.json')

if results[1] >= 0.95:
    print('\n🎯 95% accuracy target ACHIEVED!')
else:
    print(f'\n  Current accuracy: {results[1]*100:.2f}% — try training for more epochs')

---
## 📌 Cell 15 — Confusion Matrix, ROC & Precision-Recall Curves

In [ ]:
# ── Cell 15 ──────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
fig.suptitle('Model Evaluation — Test Set', fontsize=14, fontweight='bold')

# ── Confusion matrix ──
cm_vals = confusion_matrix(y_true, y_pred)
tn, fp, fn, tp = cm_vals.ravel()

# Annotate with percentage
cm_pct  = cm_vals / cm_vals.sum() * 100
annot   = np.array([[f'{v}\n({p:.1f}%)' for v, p in zip(row, prow)]
                     for row, prow in zip(cm_vals, cm_pct)])

sns.heatmap(cm_vals, annot=annot, fmt='', cmap='Blues', ax=axes[0],
            xticklabels=CLASSES, yticklabels=CLASSES,
            linewidths=0.5, linecolor='white',
            annot_kws={'size':12})
axes[0].set_title('Confusion Matrix', fontweight='bold', fontsize=12)
axes[0].set_xlabel('Predicted'); axes[0].set_ylabel('True')

spec = tn/(tn+fp+1e-8); sens = tp/(tp+fn+1e-8)
print(f'Specificity: {spec:.4f}  |  Sensitivity (Recall): {sens:.4f}')
print(f'TN={tn}  FP={fp}  FN={fn}  TP={tp}')

# ── ROC curve ──
fpr_, tpr_, _ = roc_curve(y_true, y_prob)
roc_auc_      = auc(fpr_, tpr_)
axes[1].plot(fpr_, tpr_, color='#1565c0', lw=2.5, label=f'AUC={roc_auc_:.4f}')
axes[1].plot([0,1],[0,1],'k--', lw=1, alpha=0.5, label='Random')
axes[1].fill_between(fpr_, tpr_, alpha=0.08, color='#1565c0')
axes[1].set_title('ROC Curve', fontweight='bold', fontsize=12)
axes[1].set_xlabel('False Positive Rate'); axes[1].set_ylabel('True Positive Rate')
axes[1].legend(fontsize=10); axes[1].grid(alpha=0.3)

# ── Precision-Recall curve ──
prec_arr, rec_arr, _ = precision_recall_curve(y_true, y_prob)
pr_auc = auc(rec_arr, prec_arr)
axes[2].plot(rec_arr, prec_arr, color='#6a1b9a', lw=2.5, label=f'PR-AUC={pr_auc:.4f}')
axes[2].axhline(y_true.mean(), color='gray', lw=1, linestyle='--', label='Baseline')
axes[2].fill_between(rec_arr, prec_arr, alpha=0.08, color='#6a1b9a')
axes[2].set_title('Precision-Recall Curve', fontweight='bold', fontsize=12)
axes[2].set_xlabel('Recall'); axes[2].set_ylabel('Precision')
axes[2].legend(fontsize=10); axes[2].grid(alpha=0.3)

plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/evaluation_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved evaluation_curves.png')

---
## 📌 Cell 16 — Save Final Model to Drive

In [ ]:
# ── Cell 16 ──────────────────────────────────────────────────────────────────
FINAL_PATH = f'{OUTPUT_DIR}/signature_cnn_final.keras'
model.save(FINAL_PATH)
sz = os.path.getsize(FINAL_PATH) / 1e6
print(f'✅ Model saved: {FINAL_PATH}')
print(f'   Size: {sz:.1f} MB')

# Reload sanity check
reloaded = tf.keras.models.load_model(FINAL_PATH)
dummy    = np.zeros((1,*IMG_SIZE,3), dtype='float32')
out_     = reloaded.predict(dummy, verbose=0)
print(f'✅ Reload passed — output: {out_[0][0]:.4f} (sigmoid value)')

---
## 📌 Cell 17 — Grad-CAM Helper Functions

**How Grad-CAM works with our custom CNN:**
1. A sub-model is built from `input_image` → `[last_conv output, sigmoid output]`
2. We record gradients of the predicted class score w.r.t. `last_conv` feature maps
3. Gradients are globally averaged → one weight per feature map channel
4. Feature maps are weighted and summed → raw spatial importance map  
5. ReLU removes negative activations; result is normalised to `[0,1]`
6. Upscaled to 128×128 and coloured: 🔵 low influence → 🔴 high influence

In [ ]:
# ── Cell 17 ──────────────────────────────────────────────────────────────────
# Our last conv layer is named 'last_conv' (defined in build_cnn)
GRADCAM_LAYER = 'last_conv'

# Verify the layer exists
layer_names = [l.name for l in model.layers]
assert GRADCAM_LAYER in layer_names, \
    f"'{GRADCAM_LAYER}' not found. Available: {layer_names}"
print(f'✅ Grad-CAM layer confirmed: {GRADCAM_LAYER}')
print(f'   Output shape: {model.get_layer(GRADCAM_LAYER).output_shape}')


def get_gradcam(model, img_array, layer_name=GRADCAM_LAYER):
    """
    Compute Grad-CAM heatmap for a single preprocessed image.

    Args:
        model      : trained Keras model
        img_array  : float32 numpy (1, H, W, 3) normalised to [0,1]
        layer_name : name of the conv layer to hook

    Returns:
        heatmap  : float32 numpy (H, W) in [0, 1]
        pred_val : float sigmoid value (0=genuine, 1=forged)
    """
    # Build sub-model: input → [conv_maps, prediction]
    grad_model = tf.keras.Model(
        inputs  = model.input,
        outputs = [model.get_layer(layer_name).output, model.output]
    )

    img_t = tf.cast(img_array, tf.float32)

    with tf.GradientTape() as tape:
        tape.watch(img_t)
        conv_out, preds = grad_model(img_t)
        pred_val   = preds[0, 0]                              # sigmoid scalar
        # Score for whichever class was predicted
        class_score = pred_val if float(pred_val) > 0.5 else (1.0 - pred_val)

    # Gradients of class score w.r.t. conv feature maps → (1, H, W, C)
    grads        = tape.gradient(class_score, conv_out)

    # Pool gradients spatially → importance weight per channel (C,)
    pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))

    # Weight each feature map by its importance, then sum over channels
    conv_out  = conv_out[0]                                   # (H, W, C)
    heatmap   = conv_out @ pooled_grads[..., tf.newaxis]      # (H, W, 1)
    heatmap   = tf.squeeze(heatmap)                           # (H, W)

    # ReLU → keep only positive activations
    heatmap   = tf.nn.relu(heatmap)

    # Normalise to [0, 1]
    heatmap   = heatmap / (tf.reduce_max(heatmap) + 1e-8)

    return heatmap.numpy(), float(pred_val.numpy())


def make_overlay(rgb_img, heatmap, alpha=0.45):
    """
    Blend Grad-CAM heatmap onto an RGB image.
    blue=low influence, red=high influence (Jet colormap).
    """
    H, W   = rgb_img.shape[:2]
    hm_up  = cv2.resize(heatmap, (W, H))              # upscale
    hm_u8  = np.uint8(255 * hm_up)
    hm_clr = cv2.applyColorMap(hm_u8, cv2.COLORMAP_JET)  # BGR
    hm_rgb = cv2.cvtColor(hm_clr, cv2.COLOR_BGR2RGB)
    blend  = cv2.addWeighted(
        rgb_img.astype(np.float32), 1 - alpha,
        hm_rgb.astype(np.float32),      alpha, 0
    )
    return np.clip(blend, 0, 255).astype(np.uint8)


def load_for_gradcam(img_path):
    """
    Load → preprocess → return model input tensor + display image.
    """
    proc        = preprocess_signature(img_path, size=IMG_SIZE)  # uint8 RGB
    img_display = proc.copy()
    img_array   = np.expand_dims(proc.astype('float32') / 255.0, axis=0)
    return img_array, img_display


print('✅ Grad-CAM functions ready')

---
## 📌 Cell 18 — Single Image Grad-CAM
5-panel view: Original · Preprocessed · Heatmap · Overlay · Confidence

In [ ]:
# ── Cell 18 ──────────────────────────────────────────────────────────────────
# Change random_state to see different test images
sample_row = test_df.sample(1, random_state=7).iloc[0]
IMAGE_PATH = sample_row['filepath']
TRUE_LABEL = sample_row['label']

print(f'Image      : {IMAGE_PATH}')
print(f'True label : {TRUE_LABEL}')

# Load, predict, Grad-CAM
img_array, img_disp = load_for_gradcam(IMAGE_PATH)
heatmap, pred_val   = get_gradcam(model, img_array)
overlay_img         = make_overlay(img_disp, heatmap, alpha=0.45)

is_forged  = pred_val > 0.5
pred_label = 'forged'  if is_forged else 'genuine'
confidence = pred_val  if is_forged else 1 - pred_val
correct    = pred_label == TRUE_LABEL

# Coloured heatmap panel
hm_up  = cv2.resize(heatmap, IMG_SIZE)
hm_rgb = cv2.cvtColor(cv2.applyColorMap(np.uint8(255*hm_up), cv2.COLORMAP_JET),
                      cv2.COLOR_BGR2RGB)
raw_img = np.array(Image.open(IMAGE_PATH).convert('RGB'))

# 5-panel figure
fig, axes = plt.subplots(1, 5, figsize=(22, 5))
clr     = '#2e7d32' if correct else '#c62828'
verdict = '✓ CORRECT' if correct else '✗ WRONG'
fig.suptitle(
    f'Prediction: {pred_label.upper()}  |  Confidence: {confidence*100:.1f}%'
    f'  |  True: {TRUE_LABEL.upper()}  |  {verdict}',
    fontsize=12, fontweight='bold', color=clr
)

for ax, (im, t) in zip(axes[:4], [
    (raw_img,     'Original Image'),
    (img_disp,    'Preprocessed 128×128'),
    (hm_rgb,      'Grad-CAM Heatmap\n(last_conv layer)'),
    (overlay_img, 'Heatmap Overlay  α=0.45'),
]):
    ax.imshow(im); ax.set_title(t, fontweight='bold', fontsize=9); ax.axis('off')

# Confidence bar
gc = (1 - pred_val) * 100
fc = pred_val * 100
bs = axes[4].barh(['Genuine','Forged'], [gc, fc],
                   color=['#2e7d32','#c62828'], height=0.4, edgecolor='none')
for b, v in zip(bs, [gc, fc]):
    axes[4].text(v+1, b.get_y()+b.get_height()/2,
                 f'{v:.1f}%', va='center', fontweight='bold', fontsize=10)
axes[4].set_xlim(0, 115); axes[4].set_xlabel('Confidence (%)')
axes[4].set_title('Confidence', fontweight='bold'); axes[4].grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/gradcam_single.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved gradcam_single.png')

---
## 📌 Cell 19 — Batch Grad-CAM: Genuine vs Forged

In [ ]:
# ── Cell 19 ──────────────────────────────────────────────────────────────────
N = 3   # number of genuine + number of forged to show

g_samp = test_df[test_df['label']=='genuine'].sample(N, random_state=SEED)
f_samp = test_df[test_df['label']=='forged' ].sample(N, random_state=SEED)
batch  = pd.concat([g_samp, f_samp]).reset_index(drop=True)

n_rows = len(batch)
fig, axes = plt.subplots(n_rows, 4, figsize=(18, n_rows * 4))
fig.suptitle('Batch Grad-CAM — Genuine (top) vs Forged (bottom)',
             fontsize=13, fontweight='bold', y=1.01)

for hdr, t in zip(axes[0], ['Original','Preprocessed','Heatmap','Overlay']):
    hdr.set_title(t, fontweight='bold', fontsize=11)

for ri, (_, row) in enumerate(batch.iterrows()):
    img_arr, img_d  = load_for_gradcam(row['filepath'])
    hm, pv          = get_gradcam(model, img_arr)
    ov              = make_overlay(img_d, hm, alpha=0.45)

    pl    = 'forged' if pv>0.5 else 'genuine'
    cf    = pv if pv>0.5 else 1-pv
    ok    = pl == row['label']
    clr   = '#2e7d32' if ok else '#c62828'

    raw   = np.array(Image.open(row['filepath']).convert('RGB'))
    hm_up = cv2.resize(hm, IMG_SIZE)
    hm_r  = cv2.cvtColor(cv2.applyColorMap(np.uint8(255*hm_up), cv2.COLORMAP_JET),
                         cv2.COLOR_BGR2RGB)

    for ci, im_ in enumerate([raw, img_d, hm_r, ov]):
        ax = axes[ri][ci]
        ax.imshow(im_); ax.axis('off')
        if ci == 0:
            ax.set_ylabel(
                f"True: {row['label']}\nPred: {pl}\n{cf*100:.0f}%  {'✓' if ok else '✗'}",
                fontsize=9, color=clr, fontweight='bold',
                rotation=0, labelpad=75, va='center'
            )
            ax.yaxis.set_visible(True)
            ax.tick_params(left=False, labelleft=False)

plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/gradcam_batch.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved gradcam_batch.png')

---
## 📌 Cell 20 — Confidence Threshold Tuning
Find the optimal decision threshold instead of hardcoding 0.5.

In [ ]:
# ── Cell 20 ──────────────────────────────────────────────────────────────────
thresholds = np.arange(0.1, 0.95, 0.01)
f1_scores  = []
acc_scores = []

for t in thresholds:
    yp = (y_prob > t).astype(int)
    p  = (yp * (yp == y_true)).sum() / (yp.sum() + 1e-8)
    r  = (yp * (yp == y_true)).sum() / (y_true.sum() + 1e-8)
    f1_scores.append(2*p*r/(p+r+1e-8))
    acc_scores.append((yp == y_true).mean())

best_t_f1  = thresholds[np.argmax(f1_scores)]
best_t_acc = thresholds[np.argmax(acc_scores)]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Threshold Tuning', fontsize=13, fontweight='bold')

axes[0].plot(thresholds, f1_scores,  color='#1565c0', lw=2, label='F1')
axes[0].axvline(best_t_f1, color='#c62828', lw=1.5, linestyle='--',
                label=f'Best={best_t_f1:.2f}')
axes[0].set_title('F1 Score vs Threshold', fontweight='bold')
axes[0].set_xlabel('Threshold'); axes[0].legend(); axes[0].grid(alpha=0.3)

axes[1].plot(thresholds, acc_scores, color='#2e7d32', lw=2, label='Accuracy')
axes[1].axvline(best_t_acc, color='#c62828', lw=1.5, linestyle='--',
                label=f'Best={best_t_acc:.2f}')
axes[1].axhline(0.95, color='gray', lw=1, linestyle=':', label='95% target')
axes[1].set_title('Accuracy vs Threshold', fontweight='bold')
axes[1].set_xlabel('Threshold'); axes[1].legend(); axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/threshold_tuning.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'Best threshold by F1       : {best_t_f1:.2f}  → F1={max(f1_scores):.4f}')
print(f'Best threshold by Accuracy : {best_t_acc:.2f}  → Acc={max(acc_scores)*100:.2f}%')

# Apply best accuracy threshold
BEST_THRESHOLD = best_t_acc
y_pred_best    = (y_prob > BEST_THRESHOLD).astype(int)
print(f'\nWith threshold={BEST_THRESHOLD:.2f}:')
print(classification_report(y_true, y_pred_best, target_names=CLASSES))

---
## 📌 Cell 21 — Predict on Your Own Image
Upload any signature or point to a Drive path.

In [ ]:
# ── Cell 21 ──────────────────────────────────────────────────────────────────
# ── Option A: Upload from your computer ──────────────────────────────────────
# from google.colab import files
# uploaded   = files.upload()
# MY_IMAGE   = list(uploaded.keys())[0]

# ── Option B: Drive path ─────────────────────────────────────────────────────
MY_IMAGE = test_df.sample(1, random_state=55).iloc[0]['filepath']  # ← replace
THRESHOLD = BEST_THRESHOLD   # use tuned threshold from Cell 20

# ── Predict ──────────────────────────────────────────────────────────────────
img_arr, img_d = load_for_gradcam(MY_IMAGE)
hm, pv         = get_gradcam(model, img_arr)
ov             = make_overlay(img_d, hm, alpha=0.45)

pred_label = 'FORGED'  if pv > THRESHOLD else 'GENUINE'
confidence = pv        if pv > THRESHOLD else 1 - pv

print('┌──────────────────────────────────────────────────┐')
print(f'│  Verdict      : {pred_label:<33}│')
print(f'│  Confidence   : {confidence*100:>6.2f}%                          │')
print(f'│  Raw sigmoid  : {pv:>6.4f}  (0=genuine, 1=forged)   │')
print(f'│  Threshold    : {THRESHOLD:.2f}                               │')
print('└──────────────────────────────────────────────────┘')

hm_up = cv2.resize(hm, IMG_SIZE)
hm_r  = cv2.cvtColor(cv2.applyColorMap(np.uint8(255*hm_up), cv2.COLORMAP_JET),
                     cv2.COLOR_BGR2RGB)
raw   = np.array(Image.open(MY_IMAGE).convert('RGB'))

fig, axes = plt.subplots(1, 4, figsize=(20, 5))
fig.suptitle(
    f'Verdict: {pred_label}  |  Confidence: {confidence*100:.1f}%',
    fontsize=13, fontweight='bold',
    color='#c62828' if pred_label=='FORGED' else '#2e7d32'
)
for ax, (im, t) in zip(axes, [
    (raw,     'Original'),
    (img_d,   'Preprocessed'),
    (hm_r,    'Grad-CAM Heatmap'),
    (ov,      'Overlay'),
]):
    ax.imshow(im); ax.set_title(t, fontweight='bold'); ax.axis('off')

plt.tight_layout()
plt.show()

---
## 📌 Cell 22 — Final Summary

In [ ]:
# ── Cell 22 ──────────────────────────────────────────────────────────────────
with open(f'{OUTPUT_DIR}/metrics.json') as f:
    m = json.load(f)

print('=' * 58)
print('  FINAL RESULTS — Custom CNN Forgery Detector')
print('=' * 58)
print(f'  Dataset       : GPDS (Kaggle) — {len(df)} images')
print(f'  Split         : 70 / 15 / 15')
print(f'  Architecture  : Custom 5-block CNN')
print(f'  Input size    : {IMG_SIZE[0]}×{IMG_SIZE[1]}')
print(f'  Parameters    : {model.count_params():,}')
print(f'  XAI           : Grad-CAM (last_conv layer)')
print('─' * 58)
print(f'  Test Accuracy : {m["test_accuracy"]*100:.2f}%  {"✅ target met" if m["target_met"] else ""}')
print(f'  Test AUC      : {m["test_auc"]:.4f}')
print(f'  Test Precision: {m["test_precision"]*100:.2f}%')
print(f'  Test Recall   : {m["test_recall"]*100:.2f}%')
print(f'  Test F1       : {m["test_f1"]*100:.2f}%')
print(f'  Best Threshold: {BEST_THRESHOLD:.2f}')
print('─' * 58)
print(f'  Model saved   → signature_cnn_final.keras')
print('=' * 58)
print('\n  Output files:')
for fp in sorted(Path(OUTPUT_DIR).glob('*')):
    sz = fp.stat().st_size
    u  = 'MB' if sz>1e6 else 'KB'
    print(f'  {fp.name:<45} {sz/(1e6 if sz>1e6 else 1e3):.1f} {u}')